In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

In [2]:
from models.cnn_model import cnn_model
from utils import EarlyStopping, ResizeKeepRatioPad

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
import torchvision
from torchvision import datasets
import torchvision.transforms as transforms

# Parameters

In [3]:
batch_size   = 64
epochs       = 20
hidden_size  = 16
num_output   = 10
kernel_size  = 3
stride       = 1
padding      = 1
pool_size    = 2
# channel_size = 1
input_size   = (28, 28)
patience     = 5
test_size    = 0.2
lr = 0.001

# Sample data

In [4]:
base_transform = transforms.ToTensor()

## Option 1 : MNIST dataset

In [5]:
train_dataset = torchvision.datasets.MNIST(
    root="../data",
    train=True,
    download=True,
    transform=base_transform,
)

In [6]:
test_dataset = torchvision.datasets.MNIST(
    root="../data",
    train=False,
    download=True,
    transform=base_transform,
)

## Option 2 : Local dataset

In [7]:
# full_dataset_raw = datasets.ImageFolder(
#     root="../data/MNIST/raw/",
#     transform=base_transform
# )

# test_records = int(test_size * len(full_dataset) )
# train_records = len(full_dataset) - test_records

# train_dataset, test_dataset  = random_split(full_dataset, [test_records, train_records])


# Data Processing

## Function to calculate mean and standard deviation

In [8]:
def get_mean_std(dataset, batch_size= batch_size):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    channel_sum = 0.0
    channel_sum_sq = 0.0
    num_batches = 0

    for images, _ in loader:
        
        # images: [B, C, H, W]
        channel_sum += images.mean(dim=(0, 2, 3))
        channel_sum_sq += (images ** 2).mean(dim=(0, 2, 3))
        num_batches += 1

    mean = channel_sum / num_batches
    std = (channel_sum_sq / num_batches - mean ** 2).sqrt()
    
    return mean, std

## Transform to normalized data

In [9]:
mean, std = get_mean_std( train_dataset )


final_transform = transforms.Compose([
    ResizeKeepRatioPad(input_size , fill=0),
    transforms.ToTensor(),
    transforms.Normalize(mean.tolist(), std.tolist()),
])


In [10]:
train_dataset.transform = final_transform
test_dataset.transform = final_transform

## Get number of channels

In [11]:
sample_x, sample_y = train_dataset[0]
    
channel_size = sample_x.shape[0]

In [12]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Prepare model 

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [14]:
model = cnn_model(
    channel_size=channel_size,
    input_size=input_size,
    hidden_size=hidden_size,
    num_output=num_output,
    kernel_size=kernel_size,
    stride=stride,
    padding=padding,
    pool_size=pool_size,
)

In [15]:
model.to(device)

cnn_model(
  (features): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=1, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=1, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Linear(in_features=21632, out_features=10, bias=True)
)

In [16]:
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()
early_stopping = EarlyStopping(
    patience=patience,
    path="../checkpoints/cnn_checkpoint.pt",
)

# Loop epochs and train model

In [ ]:
for epoch in range(epochs):

    # ----------------------------- Train -----------------------------
    model.train()
    train_loss = 0.0

    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        output = model(batch_x)
        loss = criterion(output, batch_y)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # ------------------- Validation -------------------

    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            output = model(batch_x)
            loss = criterion(output, batch_y)
            val_loss += loss.item()

            preds = output.argmax(dim=1)
            correct += (preds == batch_y).sum().item()
            total += batch_y.size(0)

    avg_val_loss = val_loss / len(test_loader)
    accuracy = correct / total

    print(
        f"Epoch {epoch + 1:3d} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"Accuracy: {accuracy:.4f}"
    )

    # ==================== Early Stopping Check ====================

    early_stopping(avg_val_loss, model)

    if early_stopping.early_stop:
        print("Early stopping triggered! Training stopped.")
        break

print("Training finished!")

Epoch   1 | Train Loss: 0.1172 | Val Loss: 0.0619 | Accuracy: 0.9805
Epoch   2 | Train Loss: 0.0473 | Val Loss: 0.0447 | Accuracy: 0.9844
Epoch   3 | Train Loss: 0.0307 | Val Loss: 0.0492 | Accuracy: 0.9836
